In [98]:
import pandas as pd
import datetime as dt
from acled import Acled
import numpy as np
import xgboost as xgb
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV
from sklearn.metrics import average_precision_score, classification_report, precision_recall_curve

In [99]:
acled = Acled()

INFO:acled:Access token correctly retrieved.


Train: Data from January 2018 to December 2022. This includes the 2018 Sudanese revolution but excludes the 2023 Civil War.
Test onset civil war: Data from January 2023 to December 2023 which includes the escalation of the civil war.
Test active civil war: Data from January 2024 to December 2025 which includes fluctuations in ongoing civil war.


In [100]:
countries = ["Sudan"]
start_date = "2017-07-01"
end_date = "2024-12-31"

train_start_date = "2018-01-01"
train_end_date = "2022-12-31"

onset_start_date = "2023-01-01"
onset_end_date = "2023-12-31"

active_start_date = "2024-01-01"
active_end_date = "2024-12-31"

In [101]:
all_data = acled.get_data(countries, start_date, end_date)

INFO:acled:Requesting data...
INFO:acled:Requesting data...
INFO:acled:Requesting data...
INFO:acled:Requesting data...
INFO:acled:Requesting data...
INFO:acled:All data successfully fetched.


In [102]:
def mark_conflict_events(df: pd.DataFrame) -> pd.DataFrame:
    # 1 = Conflict event (Y)
    # 0 = Non-conflict (used for features)

    acled_subevent_mapping = {
        # BATTLES (Conflict)
        "Armed clash": 1,
        "Government regains territory": 1,
        "Non-state actor overtakes territory": 1,
        # EXPLOSIONS / REMOTE VIOLENCE (Conflict)
        "Air/drone strike": 1,
        "Chemical weapon": 1,
        "Remote explosive/landmine/IED": 1,
        "Shelling/artillery/missile attack": 1,
        "Suicide bomb": 1,
        "Grenade": 1,
        # VIOLENCE AGAINST CIVILIANS (Conflict)
        "Abduction/forced disappearance": 1,
        "Attack": 1,
        "Sexual violence": 1,
        # RIOTS (Conflict)
        "Mob violence": 1,
        "Violent demonstration": 1,
        # PROTESTS (Non-conflict)
        "Excessive force against protesters": 0,
        "Peaceful protest": 0,
        "Protest with intervention": 0,
        # STRATEGIC DEVELOPMENTS (Non-conflict)
        "Agreement": 0,
        "Arrests": 0,
        "Change to group/activity": 0,
        "Disrupted weapons use": 0,
        "Headquarters or base established": 0,
        "Looting/property destruction": 0,
        "Non-violent transfer of territory": 0,
        "Other": 0,
    }
    df["conflict"] = df["sub_event_type"].apply(lambda x: acled_subevent_mapping[x])
    return df #TODO add validation

In [103]:
def create_regional_monthly_baseline(df):
    df = df.copy()
    df_grouped = (
        df.groupby(["admin2", "year_month"])["conflict"]
        .sum()
        .reset_index(name="conflict_event_count")
    )

    # Build full dataset of all regions and months
    all_regions = df["admin2"].unique()
    all_months = pd.period_range(
        df["year_month"].min(), df["year_month"].max(), freq="M"
    )
    full_index = pd.MultiIndex.from_product(
        [all_regions, all_months], names=["admin2", "year_month"]
    )

    df_grouped = (
        df_grouped.set_index(["admin2", "year_month"])
        .reindex(full_index, fill_value=0)
        .reset_index()
        .sort_values(["admin2", "year_month"])
    )

    df_grouped = df_grouped.sort_values(by=["admin2", "year_month"])

    # Calculate rolling statistics ending at the previous month (t-1)
    df_grouped["rolling_mean_6m"] = df_grouped.groupby("admin2")[
        "conflict_event_count"
    ].transform(lambda x: x.rolling(window=6, min_periods=6).mean().shift(1))

    df_grouped["rolling_std_6m"] = df_grouped.groupby("admin2")[
        "conflict_event_count"
    ].transform(lambda x: x.rolling(window=6, min_periods=6).std().shift(1))

    k = 0.5
    df_grouped["escalation_threshold"] = df_grouped["rolling_mean_6m"] + (
        k * df_grouped["rolling_std_6m"]
    )

    # Define the binary target variable (Is current conflict > historical threshold?)
    df_grouped["target_escalation"] = np.where(
        df_grouped["conflict_event_count"] > df_grouped["escalation_threshold"], 1, 0
    )

    return df_grouped

In [104]:
def pre_process_data(df):
    df = mark_conflict_events(df)

    pivot_df = pd.pivot_table(
        df,
        values="event_id_cnty",
        index=["admin2", "year_month"],
        columns=["sub_event_type"],
        aggfunc="count",
        fill_value=0,
    ).reset_index()

    pivot_df.columns = (
        pivot_df.columns.str.lower()
        .str.replace(" ", "_", regex=False)
        .str.replace("/", "_", regex=False)
        .str.replace("-", "_", regex=False)
    )

    baseline_df = create_regional_monthly_baseline(df)

    fatalities_df = (
        df.groupby(["admin2", "year_month"])["fatalities"].sum().reset_index()
    )

    combined_df = pd.merge(
        baseline_df, pivot_df, on=["admin2", "year_month"], how="left")
    combined_df = pd.merge(
        combined_df, fatalities_df, on=["admin2", "year_month"], how="left")

    event_cols = pivot_df.columns.drop(["admin2", "year_month"]).tolist()
    combined_df[event_cols] = combined_df[event_cols].fillna(0)
    combined_df["fatalities"] = combined_df["fatalities"].fillna(0)

    current_event_cols = event_cols + ["fatalities"]
    lagged_event_cols = ["rolling_mean_6m", "rolling_std_6m", "escalation_threshold"]

    combined_df[current_event_cols] = combined_df[current_event_cols].fillna(0)
    combined_df[current_event_cols] = combined_df.groupby("admin2")[current_event_cols].shift(1)

    predictor_cols = current_event_cols + lagged_event_cols
    combined_df[predictor_cols] = combined_df[predictor_cols].fillna(0)

    combined_df = combined_df.rename(columns={"admin2": "region"})
    return combined_df, predictor_cols

In [105]:
def calculate_conflict_ratio(df):
    count_0 = (df["target_escalation"] == 0).sum()
    count_1 = (df["target_escalation"] == 1).sum()
    ratio = count_0 / count_1

    return {"non-escalation": count_0, "escalation": count_1, "ratio": ratio}

In [106]:
def split_data(df, predictor_cols, target_col, start_date, end_date):
    split_df = df[
        (df["year_month"] >= start_date)
        & (df["year_month"] <= end_date)
    ].copy()

    y = split_df[target_col].copy()
    X = split_df[predictor_cols].copy()

    return split_df, y, X

In [107]:
processed_df, predictor_cols = pre_process_data(all_data)

train_df, train_y, train_X = split_data(processed_df, predictor_cols, "target_escalation", train_start_date, train_end_date)

onset, onset_y, onset_X = split_data(processed_df, predictor_cols, "target_escalation", onset_start_date, onset_end_date)


In [96]:


# train_df = processed_df[
#     (processed_df["year_month"] >= train_start_date)
#     & (processed_df["year_month"] <= train_end_date)
# ].copy()
#
# onset_df = processed_df[
#     (processed_df["year_month"] >= onset_start_date)
#     & (processed_df["year_month"] <= onset_end_date)
# ].copy()
#
# active_df = processed_df[
#     (processed_df["year_month"] >= active_start_date)
#     & (processed_df["year_month"] <= active_end_date)
# ].copy()
#
# ratios = calculate_conflict_ratio(train_df)
#
# y_train = train_df["target_escalation"].copy()
# y_onset = onset_df["target_escalation"].copy()
# y_active = active_df["target_escalation"].copy()
#
# X_train = train_df[predictor_cols].copy()
# X_onset = onset_df[predictor_cols].copy()
# X_active = active_df[predictor_cols].copy()